## **Select and Validate the Masking Token**

This notebook documents the selection and validation of the replacement token used for within-post masking interventions.

The final masking design distinguishes between two operations:

1. **Target masking:** the supplied target value is replaced by an empty string (`""`), so that the target identity is genuinely absent while the surrounding input structure remains unchanged.
2. **Within-post masking:** selected spans inside the context posts are replaced by one constant synthetic token. The same token is used for the actual interventions and their matched controls.

The replacement token should satisfy the following requirements:

- it does not occur as an ordinary standalone token in the original training inputs;
- it is absent from the frozen TF-IDF vocabulary;
- it is represented as exactly one RoBERTa token sentence-initially;
- it is represented as exactly one RoBERTa token after whitespace.

Eligibility is determined using only these technical properties. The final replacement token is then fixed from the eligible set without using model predictions, intervention effects, or evaluation results.

After the token has been selected and frozen, its absence from the human test inputs is verified separately.

---

### **1. Setup**

Load the fixed training set, the frozen TF-IDF model, and the pretrained RoBERTa tokenizer. The human test set is loaded only after the replacement token has been selected and frozen.

The TF-IDF model is loaded only to inspect its fitted analyzer and vocabulary. It is not refitted or modified.

In [1]:
from pathlib import Path
import re

import numpy as np
import joblib
import pandas as pd
from transformers import AutoTokenizer

/opt/homebrew/Caskroom/miniforge/base/envs/nlp-transformers/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data/preprocessed")
MODEL_DIR = Path("../models/tfidf_logreg")

TRAIN_PATH = DATA_DIR / "train.parquet"
HUMAN_TEST_PATH = DATA_DIR / "human_test.parquet"
TFIDF_MODEL_PATH = MODEL_DIR / "model.joblib"

Load data, model, tokenizer

In [3]:
train = pd.read_parquet(TRAIN_PATH)

tfidf_logreg = joblib.load(TFIDF_MODEL_PATH)
vectorizer = tfidf_logreg.named_steps["tfidf"]

roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [4]:
print(f"Training examples: {len(train):,}")
print(f"TF-IDF vocabulary size: {len(vectorizer.vocabulary_):,}")

Training examples: 12,834
TF-IDF vocabulary size: 203,461


---

### **2. Problems With the Initial Placeholder Design**

Earlier versions used placeholders such as `[CANDIDATE]` and `[REMOVED]`.

For the TF-IDF model, bracketed placeholders are processed by the fitted analyzer. If components such as `candidate` or `removed` already occur in the frozen vocabulary, masking can introduce learned features instead of only removing information.

Arbitrary synthetic strings may also be split into several RoBERTa subword tokens, which can introduce unnecessary changes to sequence length and truncation behavior.

We therefore search for a replacement token that avoids these problems.

In [5]:
old_placeholders = [
    "[CANDIDATE]",
    "[REMOVED]",
    "[TARGET_REMOVED]",
    "[XQZ]",
]

for placeholder in old_placeholders:
    tfidf_features = vectorizer.build_analyzer()(placeholder)

    tfidf_overlap = [
        feature
        for feature in tfidf_features
        if feature in vectorizer.vocabulary_
    ]

    print("Placeholder:", placeholder)
    print("TF-IDF analyzer:", tfidf_features)
    print("TF-IDF vocabulary overlap:", tfidf_overlap)
    print(
        "RoBERTa initial:",
        roberta_tokenizer.tokenize(placeholder),
    )
    print(
        "RoBERTa after space:",
        roberta_tokenizer.tokenize(" " + placeholder),
    )
    print()

Placeholder: [CANDIDATE]
TF-IDF analyzer: ['candidate']
TF-IDF vocabulary overlap: ['candidate']
RoBERTa initial: ['[', 'C', 'AND', 'ID', 'ATE', ']']
RoBERTa after space: ['Ġ[', 'C', 'AND', 'ID', 'ATE', ']']

Placeholder: [REMOVED]
TF-IDF analyzer: ['removed']
TF-IDF vocabulary overlap: ['removed']
RoBERTa initial: ['[', 'REM', 'OV', 'ED', ']']
RoBERTa after space: ['Ġ[', 'REM', 'OV', 'ED', ']']

Placeholder: [TARGET_REMOVED]
TF-IDF analyzer: ['target_removed']
TF-IDF vocabulary overlap: []
RoBERTa initial: ['[', 'T', 'ARGET', '_', 'REM', 'OV', 'ED', ']']
RoBERTa after space: ['Ġ[', 'T', 'ARGET', '_', 'REM', 'OV', 'ED', ']']

Placeholder: [XQZ]
TF-IDF analyzer: ['xqz']
TF-IDF vocabulary overlap: []
RoBERTa initial: ['[', 'X', 'Q', 'Z', ']']
RoBERTa after space: ['Ġ[', 'X', 'Q', 'Z', ']']



---

### **3. Candidate-Token Criteria**

Candidate replacement strings are evaluated only according to tokenizer, frozen-vocabulary, and corpus-presence properties.

Because RoBERTa uses whitespace-sensitive tokenization, the sentence-initial and whitespace-prefixed representations may correspond to different vocabulary entries. The requirement is therefore that each representation consists of exactly one token, not that both positions use the same token ID.

In [6]:
def is_single_roberta_token(token):
    initial = roberta_tokenizer.tokenize(token)
    after_space = roberta_tokenizer.tokenize(" " + token)

    return len(initial) == 1 and len(after_space) == 1

In [7]:
def has_no_tfidf_overlap(token):
    features = vectorizer.build_analyzer()(token)

    overlap = [
        feature
        for feature in features
        if feature in vectorizer.vocabulary_
    ]

    return len(overlap) == 0

### **4. Generate the Tokenizer- and Vocabulary-Eligible Candidate Pool**

We first identify strings that satisfy the model-specific technical requirements. At this stage, candidates are filtered only according to the pretrained RoBERTa tokenizer and the frozen TF-IDF vocabulary.

Candidate strings were restricted to lowercase alphabetic tokens with at least four characters. This excludes punctuation-containing tokens, capitalization-specific forms, and very short subword fragments. The restriction is only used to define a practical candidate pool and does not make any remaining token uniquely preferable.

Corpus occurrence is checked separately in the next step.

In [8]:
roberta_vocab = roberta_tokenizer.get_vocab()

eligible_candidates = []

for vocab_token in roberta_vocab:
    if not vocab_token.startswith("Ġ"):
        continue

    candidate = vocab_token[1:]

    if not candidate.isalpha():
        continue

    if not candidate.islower():
        continue

    if len(candidate) < 4:
        continue

    if candidate not in roberta_vocab:
        continue

    if not is_single_roberta_token(candidate):
        continue

    if not has_no_tfidf_overlap(candidate):
        continue

    eligible_candidates.append(candidate)

eligible_candidates = sorted(set(eligible_candidates))

print(len(eligible_candidates))

534


### **5. Exclude Candidates Present in the Training Data**

The replacement token should not already occur as an ordinary standalone token in the original training inputs.

We therefore collect the lowercase alphabetic surface tokens occurring in the supplied targets and retrieved context posts and exclude any eligible candidate that is already present.

Only the training data are used during token selection. The human test set is not used to determine the replacement token. After the token has been selected and frozen, its absence from the human test inputs is verified separately as a post-selection validation check.

In [9]:
def collect_standalone_alpha_tokens(df):
    token_pattern = re.compile(r"(?<!\w)[a-z]+(?!\w)")

    observed_tokens = set()

    for target in df["TargetEntity"]:
        if isinstance(target, str):
            observed_tokens.update(
                token_pattern.findall(target.lower())
            )

    for context_posts in df["ContextPosts"]:
        if isinstance(context_posts, np.ndarray):
            context_posts = context_posts.tolist()

        if not isinstance(context_posts, (list, tuple)):
            continue

        for post in context_posts:
            if not isinstance(post, dict):
                continue

            text = post.get("Content")

            if isinstance(text, str):
                observed_tokens.update(
                    token_pattern.findall(text.lower())
                )

    return observed_tokens

In [10]:
train_surface_tokens = collect_standalone_alpha_tokens(train)

print(f"Unique lowercase alphabetic tokens in training: {len(train_surface_tokens):,}")

Unique lowercase alphabetic tokens in training: 36,249


In [11]:
training_absent_candidates = [
    candidate
    for candidate in eligible_candidates
    if candidate not in train_surface_tokens
]

print(
    f"Eligible candidates before training check: "
    f"{len(eligible_candidates):,}"
)
print(
    f"Eligible candidates absent from training: "
    f"{len(training_absent_candidates):,}"
)

Eligible candidates before training check: 534
Eligible candidates absent from training: 434


In [12]:
"requ" in training_absent_candidates

True

In [13]:
candidate = "requ"

print("Candidate:", candidate)
print("Present in training:", candidate in train_surface_tokens)
print("TF-IDF analyzer:", vectorizer.build_analyzer()(candidate))
print("TF-IDF vocabulary overlap:", [
    feature
    for feature in vectorizer.build_analyzer()(candidate)
    if feature in vectorizer.vocabulary_
])
print("RoBERTa initial:", roberta_tokenizer.tokenize(candidate))
print("RoBERTa after space:", roberta_tokenizer.tokenize(" " + candidate))

Candidate: requ
Present in training: False
TF-IDF analyzer: ['requ']
TF-IDF vocabulary overlap: []
RoBERTa initial: ['requ']
RoBERTa after space: ['Ġrequ']


### **6. Freeze the Replacement Token**

The preceding procedure identifies a set of technically eligible replacement tokens rather than a unique optimal token. From this set, requ is fixed as one eligible candidate for all within-post masking conditions.

The choice of `requ` is not based on model predictions, intervention effects, or evaluation performance. Its suitability follows from the predefined technical criteria: it is absent from the training inputs, absent from the frozen TF-IDF vocabulary, and represented as one RoBERTa token both sentence-initially and after whitespace.

In [14]:
CONTEXT_MASK_TOKEN = "requ"

assert CONTEXT_MASK_TOKEN in training_absent_candidates

print(f"Frozen context masking token: {CONTEXT_MASK_TOKEN}")

Frozen context masking token: requ


### **7. Post-Selection Validation on the Human Test Set**

The masking token was selected and frozen without using the human test inputs. As a final contamination check, we verify that requ also does not occur as an ordinary standalone token in the held-out human test inputs.

This check does not alter the token choice.

In [15]:
human_test = pd.read_parquet(HUMAN_TEST_PATH)

test_surface_tokens = collect_standalone_alpha_tokens(human_test)

print(f"Human test examples: {len(human_test):,}")
print(
    f"`{CONTEXT_MASK_TOKEN}` present in human test:",
    CONTEXT_MASK_TOKEN in test_surface_tokens,
)

Human test examples: 890
`requ` present in human test: False


### **8. Final Masking Design**

The masking design is therefore frozen as follows:

- **Target masking:** replace the supplied target value with an empty string (`""`), leaving the surrounding input structure unchanged.
- **Candidate-mention masking:** replace selected spans inside context posts with `requ`.
- **Candidate matched controls:** replace selected control spans with `requ`.
- **Lexical-cue masking:** replace selected spans with `requ`.
- **Lexical-cue matched controls:** replace selected control spans with `requ`.

`requ` is treated as a controlled synthetic replacement token, not as a semantically neutral token.

Target swapping and leave-one-post-out are separate interventions and do not use the masking token.